# Metrics Feature Engineering & Anomaly Detection

This notebook demonstrates:
1. Loading Prometheus metrics exported via `fetch_prometheus.py`
2. Exploratory data analysis
3. Feature engineering (rolling stats, time features)
4. Training IsolationForest and visualising anomalies

**THESIS NOTE**: Use this notebook for your chapter on feature engineering.
Document each design decision and justify it with domain knowledge.

In [ ]:
import sys
sys.path.insert(0, '../anomaly_detection')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

from metrics_isolation_forest import engineer_features, FEATURE_COLS

plt.rcParams['figure.figsize'] = (14, 5)
sns.set_style('whitegrid')
print('Imports OK')

## 1. Load Data

In [ ]:
# ADJUST PATH as needed
df = pd.read_parquet('../data/metrics.parquet')
print(f'Shape: {df.shape}')
df.head()

## 2. Exploratory Data Analysis

In [ ]:
df.describe()

In [ ]:
fig, axes = plt.subplots(len(FEATURE_COLS), 1, figsize=(14, 3*len(FEATURE_COLS)), sharex=True)
for ax, col in zip(axes, FEATURE_COLS):
    if col in df.columns:
        ax.plot(df.index, df[col], linewidth=0.8)
        ax.set_ylabel(col)
plt.suptitle('Raw Metrics Over Time')
plt.tight_layout()
plt.show()

## 3. Feature Engineering

In [ ]:
features = engineer_features(df)
print(f'Feature matrix shape: {features.shape}')
features.head()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 10))
corr = features.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='coolwarm', center=0, linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 4. Train IsolationForest

In [ ]:
# Set your train/test boundary
TRAIN_END = pd.Timestamp('2024-06-01T08:00:00Z')  # ADJUST
CONTAMINATION = 0.02  # Expected fraction of anomalies in training data

mask_train = features.index < TRAIN_END
X_train = features[mask_train].values
X_all = features.values

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_all_s = scaler.transform(X_all)

clf = IsolationForest(n_estimators=200, contamination=CONTAMINATION, random_state=42)
clf.fit(X_train_s)

scores = clf.decision_function(X_all_s)
labels = clf.predict(X_all_s)

result = df.copy()
result['anomaly_score'] = scores
result['is_anomaly'] = (labels == -1).astype(int)

print(f'Anomalies: {result.is_anomaly.sum()} ({100*result.is_anomaly.mean():.1f}%)')

## 5. Visualise Results

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
for ax, col in zip(axes, ['cpu_busy', 'p95_latency', 'error_ratio']):
    if col not in result.columns:
        continue
    ax.plot(result.index, result[col], linewidth=0.8, label=col)
    anom = result[result['is_anomaly']==1]
    ax.scatter(anom.index, anom[col], color='red', s=12, zorder=5, label='anomaly')
    ax.legend()
    ax.set_ylabel(col)
plt.suptitle('Detected Anomalies')
plt.tight_layout()
plt.show()

In [ ]:
# THESIS NOTE: Add more cells below for:
# - Effect of contamination parameter sweep
# - Feature importance (permutation importance)
# - Comparison with ground truth incident windows